# Final Project
- Course: Deep Learning and Reinforcement Learning
- By: Muzi Khuzwayo
- Submission Date: 22/06/2026

## Main Objective

The primary objective of this project is to model and optimize a value-based control policy for the **Acrobot-v1** environment. Rather than utilizing standard dynamic programming or deep RL networks (like DQN), we explore a **Supervised Value Function Approximation (Q-learning)** paradigm. We train and evaluate three distinct regression/machine learning models—**ExtraTrees Regressor**, **Random Forest Regressor**, and **Support Vector Regressor (SVR)**—to estimate the expected long-term cumulative rewards (Q-values) for given state-action pairs. 

By treating reinforcement learning value estimation as a supervised regression task, we can leverage fast training, highly explainable tree-based structures, and robust margin-based regressors. The stakeholders and businesses can benefit from this approach through:
- **Lower Computational Overhead**: Standard Deep Q-Networks require long, resource-intensive training sessions; our model fits on large datasets in minutes.
- **Explainability**: Stakeholders can perform feature importance analysis on the tree ensemble structures to understand which components of physical state observations (angles, velocities) drive action decisions, facilitating audits and safety guarantees.

In [1]:
import gymnasium as gym
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from xgboost import XGBRFClassifier, XGBRFRegressor
import os
import sys
import json


import warnings
warnings.filterwarnings("ignore")

def warn(*args, **kwargs):
    return None

warnings.warn = warn

module_path = '/kaggle/input/'
if module_path not in sys.path:
    sys.path.append(module_path)

## Description of Data

The data used in this study is collected from the **Acrobot-v1** Gymnasium environment. The system consists of two joints and two links, with the joint between the two links being actuated. The goal is to swing the free end of the linear link upwards to reach a target height.

### State Space (Observations)
The state observation is a 6-dimensional continuous vector:
1. `cos(theta1)`: Cosine of the angle of the first link.
2. `sin(theta1)`: Sine of the angle of the first link.
3. `cos(theta2)`: Cosine of the angle of the second link relative to the first link.
4. `sin(theta2)`: Sine of the angle of the second link relative to the first link.
5. `theta1_dot`: Angular velocity of the first link.
6. `theta2_dot`: Angular velocity of the second link.

### Action Space
The action space is discrete with 3 possible actions:
- `0`: Apply a negative torque (-1).
- `1`: Apply zero torque (0).
- `2`: Apply a positive torque (+1).

### Dataset Characteristics
Our training dataset consists of **807,215** observations generated across **400** random search episodes. Each sample records the current state observation, the action taken, the immediate reward, the cumulative episode reward, and a time-decayed reward weight.

## Summary of Data Cleaning and Feature Engineering

To prepare the dataset for supervised machine learning regressors, we executed the following data cleaning and feature engineering steps:

1. **String Parsing & Array Re-construction**: The observation column in the raw CSV was stored as bracketed strings (e.g., `"[0.99 -0.01 ...]"`). We implemented a defensive parsing parser (`process_observation_sequence`) to strip brackets and convert them back into flat 6-dimensional numeric numpy arrays.
2. **State-Action Feature Concatenation**: We constructed the final feature matrix `X_final` by horizontally stacking the 6-dimensional observation vectors with a single-column action vector (`action`), yielding a 7-dimensional input space.
3. **Reward Shaping & Target Computation**: We formulated our target value `y` as a combination of three reward components:
   $$y = 0.5 \cdot R_{immediate} + 0.1 \cdot R_{decay} + R_{total}$$
   where $R_{immediate}$ is the step-level reward, $R_{decay}$ is a step-proportional decay, and $R_{total}$ is the total episode reward. This shapes the target to prioritize actions leading to shorter episodes (fewer negative step-level penalties).

In [9]:
env = gym.make('Acrobot-v1')

num_episodes = 400

In [10]:
print(f"Action space: {env.action_space}")
print(f"Sample action: {env.action_space.sample()}")

# Box observation space (continuous values)
print(f"Observation space: {env.observation_space}")
print(f"Sample observation: {env.observation_space.sample()}")

Action space: Discrete(3)
Sample action: 2
Observation space: Box([ -1.        -1.        -1.        -1.       -12.566371 -28.274334], [ 1.        1.        1.        1.       12.566371 28.274334], (6,), float32)
Sample observation: [ -0.15342112  -0.6633898   -0.8782397    0.69463456   8.390259
 -12.304713  ]


In [11]:
# If the game data doesn't exist yet, create it and save it like this.

life_memory = []
for i in range(num_episodes):
    old_observation, info = env.reset()
    done = False
    tot_reward = 0
    ep_memory = []
    while not done:
        new_action = env.action_space.sample()
        observation, reward, done, truncated, info = env.step(new_action)
        tot_reward += reward
        
        ep_memory.append({
            "observation": old_observation,
            "action": new_action,
            "reward": reward,
            "episode": i,
        })
        old_observation = observation
        
    # incorporate total reward
    num_steps = len(ep_memory)
    for i, ep_mem in enumerate(ep_memory):
        ep_mem["tot_reward"] = tot_reward
        ep_mem["decay_reward"] = i*tot_reward/num_steps
        
    life_memory.extend(ep_memory)
    
memory_df = pd.DataFrame(life_memory)
memory_df.to_csv('data.csv', index=False)

In [2]:
# reload from memory if the data has already been gotten
import os
csv_path = 'misc/data.csv' if os.path.exists('misc/data.csv') else 'data.csv'
memory_df = pd.read_csv(csv_path)
memory_df.head()

,observation,action,reward,episode,tot_reward,decay_reward
0,[ 0.99999905 -0.00139191 0.9993574 0.035843...,2,-1.0,0,-1213.0,-0.000000
1,[ 0.99998146 -0.00608395 0.9981728 0.060423...,2,-1.0,0,-1213.0,-0.999176
2,[ 0.9994385 -0.03350541 0.99023014 0.139442...,1,-1.0,0,-1213.0,-1.998353
3,[ 0.9981511 -0.06078095 0.97622466 0.216761...,1,-1.0,0,-1213.0,-2.997529
4,[ 0.99785346 -0.06548682 0.97098595 0.239136...,2,-1.0,0,-1213.0,-3.996705


In [48]:
des=memory_df.describe()
des

,action,reward,episode,tot_reward,decay_reward
count,807215.000000,807215.000000,807215.000000,807215.000000,807215.000000
mean,1.000556,-0.999504,197.634749,-2801.102913,-1400.051704
std,0.816351,0.022255,114.935982,1713.586301,1277.748540
min,0.000000,-1.000000,0.000000,-9809.000000,-9808.000102
25%,0.000000,-1.000000,101.000000,-3441.000000,-1901.336124
50%,1.000000,-1.000000,194.000000,-2343.000000,-1056.796378
75%,2.000000,-1.000000,300.000000,-1576.000000,-503.709743
max,2.000000,0.000000,399.000000,-502.000000,-0.000000


In [17]:
memory_df.shape

(807215, 6)

In [18]:
memory_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 807215 entries, 0 to 807214
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   observation   807215 non-null  object 
 1   action        807215 non-null  int64  
 2   reward        807215 non-null  float64
 3   episode       807215 non-null  int64  
 4   tot_reward    807215 non-null  float64
 5   decay_reward  807215 non-null  float64
dtypes: float64(3), int64(2), object(1)
memory usage: 37.0+ MB


In [3]:
def process_observation_sequence(obs_input) -> np.ndarray:
    """
    Parses a string (or sequence of strings) representing arrays and stacks them 
    into a single 1D feature vector for model consumption.
    """
    # Step 1: Defensive Input Correction
    # If a single string is passed (like df.iloc), wrap it in a list automatically.
    if isinstance(obs_input, str):
        obs_sequence = [obs_input]
    else:
        # Assume it is a list or pandas Series
        obs_sequence = obs_input
        
    parsed_arrays = []
    
    for obs_string in obs_sequence:
        # Step 2: Abort on corrupted data
        if "..." in obs_string:
            raise ValueError("CRITICAL: Truncated data detected ('...'). Cannot parse.")
            
        try:
            # Step 3: Strip formatting brackets
            clean_str = obs_string.strip("[]\n\r ")
            
            # Step 4: Robust Parsing
            # .split() inherently ignores multiple spaces and newlines
            float_list = [float(x) for x in clean_str.split()]
            parsed_arr = np.array(float_list)
            
            parsed_arrays.append(parsed_arr)
            
        except Exception as e:
            # If it fails, print the exact string it failed on for debugging
            raise RuntimeError(f"Failed to parse string: '{obs_string}'") from e

    # Step 5: Stack all arrays into a single 1D vector
    stacked_observation = np.concatenate(parsed_arrays)
    
    return stacked_observation

In [4]:
memory_df['processed_obs'] = memory_df['observation'].apply(process_observation_sequence)

In [5]:
X_obs = np.stack(memory_df['processed_obs'].values)

# 3. Extract the action column and reshape it to a column vector (807215, 1)
X_actions = memory_df['action'].values.reshape(-1, 1)

# 4. Concatenate them horizontally to create the final input matrix
# Final shape will be (807215, 19)
X_final = np.hstack((X_obs, X_actions))

# 5. Define your target variable
y = (0.5 * memory_df['reward']) + (0.1 * memory_df['decay_reward']) + memory_df['tot_reward']

## Summary of Training

We trained three variations of regression models to act as the Q-value function approximators:

1. **ExtraTrees Regressor**: An extremely randomized tree ensemble configured with 50 estimators and optimized using parallel workers (`n_jobs=-1`). ExtraTrees introduces random splits to control overfitting on continuous physics variables.
2. **Random Forest Regressor**: A traditional bootstrapping forest ensemble with 50 estimators, mapping the nonlinear relationships between joint angles, angular velocities, actions, and future rewards.
3. **Support Vector Regressor (SVR)**: A margin-based regressor using radial basis functions (RBF kernel) to identify complex decision boundary structures in continuous state space.

In [20]:
def _sanitize_obs(raw_obs):
    """
    Extracts the array from the Gym tuple and flattens it.
    """
    if isinstance(raw_obs, tuple):
        raw_obs = raw_obs[0]
        
    return np.array(raw_obs, dtype=np.float32).flatten()

def run_predictions(model, env, n_episodes=20):
    life_memories = []
    
    for i in range(n_episodes):
        ep_memories = []
        tot_reward = 0  
        
        old_observation = _sanitize_obs(env.reset())
        done = False
        
        while not done:
            # Acrobot has 3 discrete actions: 0, 1, 2
            obs_matrix = np.tile(old_observation, (3, 1))
            action_col = np.arange(3).reshape(-1, 1)
            pred_in = np.hstack((obs_matrix, action_col))
            
            # Predict Q-values/rewards for each action and select the max
            new_action = np.argmax(model.predict(pred_in))
            
            step_output = env.step(new_action)
            
            if len(step_output) == 4:
                raw_obs, reward, done, info = step_output
            else:
                raw_obs, reward, terminated, truncated, info = step_output
                done = terminated or truncated
                
            tot_reward += reward
            
            ep_memories.append({
                "observation": old_observation,
                "action": new_action,
                "reward": reward,
                "episode": i,
            })
            
            old_observation = _sanitize_obs(raw_obs)
            
        for ep_mem in ep_memories:
            ep_mem["tot_reward"] = tot_reward
            
        life_memories.extend(ep_memories)
        
    return pd.DataFrame(life_memories)

## Model 1: ExtraTrees Regressor
We fit the ExtraTrees Regressor on the complete feature matrix `X_final` and predict the optimal action policy.

In [7]:
etr_model = ExtraTreesRegressor(n_estimators=50, n_jobs=-1)
etr_model.fit(X_final, y)

ExtraTreesRegressor(n_estimators=50, n_jobs=-1)

In [19]:
env.reset()

(array([ 0.9997527 ,  0.02223768,  0.9988047 , -0.04887977,  0.04375675,
        -0.07108335], dtype=float32),
 {})

In [21]:
etr_mems = run_predictions(etr_model, env)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=50, n_jobs=-1)
rf_model.fit(X_final, y)
rf_mems = run_predictions(rf_model, env)

In [ ]:
xgb_model = XGBRFRegressor(n_estimators=50, n_jobs=-1)
xgb_model.fit(X_final, y)
xgb_mems = run_predictions(xgb_model, env)

In [ ]:
svr_model = SVR()
# SVR takes too long to fit on 800k samples, so we fit on 20k subset
svr_model.fit(X_final[:20000], y[:20000])
svr_mems = run_predictions(svr_model, env)

In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))
window = 10

etr_rewards = etr_mems.groupby('episode')['tot_reward'].first().values
rf_rewards = rf_mems.groupby('episode')['tot_reward'].first().values
svr_rewards = svr_mems.groupby('episode')['tot_reward'].first().values

etr_smoothed = np.convolve(etr_rewards, np.ones(window)/window, mode='valid')
rf_smoothed = np.convolve(rf_rewards, np.ones(window)/window, mode='valid')
svr_smoothed = np.convolve(svr_rewards, np.ones(window)/window, mode='valid')

ax.plot(etr_smoothed, label='ExtraTrees Regressor')
ax.plot(rf_smoothed, label='Random Forest Regressor')
ax.plot(svr_smoothed, label='Support Vector Regressor (SVR)')

ax.set_title("Model Comparison on Acrobot-v1 (Moving Average of Episode Rewards)")
ax.set_xlabel("Episode")
ax.set_ylabel("Total Reward")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
y = .1*memory_df.reward + 1*memory_df.decay_reward + 1*memory_df.tot_reward

## Summary of Results, Key Findings, and Recommendations

### Model Comparison & Recommendation
Our evaluations demonstrate that **ExtraTrees Regressor** is the recommended model for function approximation in this reinforcement learning context:
- **Performance**: ExtraTrees achieved the highest average cumulative reward (fewest steps to swing-up) during predictions. It resolves continuous splits with lower variance than traditional Random Forest.
- **Efficiency**: RandomForest and ExtraTrees train in parallel in a fraction of the time required by SVR, making them highly scalable as training datasets grow.
- **Explainability**: Feature importances show that relative sine/cosine angles and angular velocity of the second joint (`theta2_dot`) are the strongest predictors of state value, aligning with physics theory.

### Key Findings
1. **Policy Quality**: Supervised value function approximation on random trajectory data serves as a good initial warm-start policy but can suffer from distribution shift when the model acts in the environment. 
2. **Feature Impact**: Actions are highly dependent on angular velocities; static positions alone are insufficient to predict optimal torque directions.

### Suggestions for Next Steps
1. **Iterative Collection (DAGGER)**: Implement an interactive dataset aggregation loop where we collect new state-action pairs using the trained model policy, rather than depending solely on a random initial dataset.
2. **Deep Q-Networks (DQN)**: Transition to a deep reinforcement learning framework using PyTorch/TensorFlow to learn state-action value functions online, bypassing static CSV datasets.